
# 10 — Model Error Analysis and Feature Importance

## Objectives
1. Identify systematic prediction errors.
2. Analyse underprediction and overprediction.
3. Evaluate errors across player groups and actual run categories.
4. Investigate residual distributions and outliers.
5. Calculate feature importance using appropriate methods.
6. Compare model interpretability and generalisation.
7. Document limitations and recommendations.

## Important
- Reuse existing trained models and saved predictions from Notebook 09.
- Do not retrain models unnecessarily.
- Do not modify the original datasets.
- Keep validation and test results separate.
- Do not use test data to select or tune models.


In [1]:
from pathlib import Path

PROJECT_ROOT = Path.cwd()

# Locate project root if the notebook was opened from notebooks/
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

print("PROJECT ROOT:", PROJECT_ROOT)
print("EXISTS:", PROJECT_ROOT.exists())

artifact_dirs = [
    "models",
    "artifacts",
    "reports",
    "outputs",
    "data/processed",
]

extensions = {".joblib", ".pkl", ".pickle", ".csv", ".json"}

print("\nSAVED MODEL AND RESULT FILES")
print("=" * 65)

for folder in artifact_dirs:
    directory = PROJECT_ROOT / folder

    if not directory.exists():
        print(f"\n{folder}: DIRECTORY NOT FOUND")
        continue

    files = [
        path for path in directory.rglob("*")
        if path.is_file() and path.suffix.lower() in extensions
    ]

    print(f"\n{folder}: {len(files)} relevant files")

    for path in files[:30]:
        print(f"  {path.relative_to(PROJECT_ROOT)}")

    if len(files) > 30:
        print(f"  ... and {len(files) - 30} more")

PROJECT ROOT: d:\Git\Personal\ipl-player-performance-v2
EXISTS: True

SAVED MODEL AND RESULT FILES

models: 0 relevant files

artifacts: 0 relevant files

reports: DIRECTORY NOT FOUND

outputs: DIRECTORY NOT FOUND

data/processed: 18 relevant files
  data\processed\deliveries.csv
  data\processed\etl_manifest.json
  data\processed\innings.csv
  data\processed\matches.csv
  data\processed\features\batting_all_model_comparison.csv
  data\processed\features\batting_baseline_comparison.csv
  data\processed\features\batting_baseline_results.csv
  data\processed\features\batting_complete_model_comparison.csv
  data\processed\features\batting_model_comparison.csv
  data\processed\features\batting_model_dataset.csv
  data\processed\features\batting_next_match_features.csv
  data\processed\features\bowling_model_dataset.csv
  data\processed\features\bowling_next_match_features.csv
  data\processed\features\feature_schema.json
  data\processed\player_match\batting_player_match_regular.csv
  data

In [2]:
import pandas as pd
from pathlib import Path

FEATURE_DIR = PROJECT_ROOT / "data" / "processed" / "features"

result_files = [
    "batting_all_model_comparison.csv",
    "batting_baseline_comparison.csv",
    "batting_baseline_results.csv",
    "batting_complete_model_comparison.csv",
    "batting_model_comparison.csv",
]

print("=" * 75)
print("EXISTING MODEL RESULT FILE AUDIT")
print("=" * 75)

for filename in result_files:
    file_path = FEATURE_DIR / filename

    print(f"\n{'=' * 65}")
    print("FILE:", filename)

    if not file_path.exists():
        print("STATUS: NOT FOUND")
        continue

    result_df = pd.read_csv(file_path)

    print("STATUS: FOUND")
    print("Shape:", result_df.shape)
    print("Columns:", result_df.columns.tolist())

    display(result_df.head(10))

EXISTING MODEL RESULT FILE AUDIT

FILE: batting_all_model_comparison.csv
STATUS: FOUND
Shape: (18, 4)
Columns: ['Model', 'MAE', 'RMSE', 'R2']


,Model,MAE,RMSE,R2
0,Gradient Boosting 200,15.932161,21.423562,0.117053
1,Gradient Boosting 100,15.953633,21.424681,0.116961
2,Gradient Boosting Depth 3,15.961878,21.419572,0.117382
3,Random Forest Depth 10,16.002567,21.475437,0.112772
4,Ridge Alpha 1000,16.078629,21.447241,0.115101
5,Ridge Alpha 3000,16.085646,21.445985,0.115204
6,Ridge Alpha 100,16.105206,21.452342,0.114680
7,Ridge Alpha 10000,16.118473,21.457059,0.114290
8,Ridge Alpha 10,16.158372,21.463845,0.113730
9,Random Forest Depth 15,16.162721,21.562778,0.105541



FILE: batting_baseline_comparison.csv
STATUS: FOUND
Shape: (6, 4)
Columns: ['Model', 'MAE', 'RMSE', 'R2']


,Model,MAE,RMSE,R2
0,Ridge Alpha 100,16.105206,21.452342,0.114680
1,Ridge Alpha 10,16.158372,21.463845,0.113730
2,Ridge Alpha 1,16.188406,21.477762,0.112580
3,Linear Regression,16.193309,21.480407,0.112362
4,Dummy Median,16.683064,24.253256,-0.131595
5,Dummy Mean,17.361069,22.835217,-0.003140



FILE: batting_baseline_results.csv
STATUS: FOUND
Shape: (2, 4)
Columns: ['Model', 'MAE', 'RMSE', 'R2']


,Model,MAE,RMSE,R2
0,Dummy Mean,17.361069,22.835217,-0.003140
1,Dummy Median,16.683064,24.253256,-0.131595



FILE: batting_complete_model_comparison.csv
STATUS: FOUND
Shape: (21, 4)
Columns: ['Model', 'MAE', 'RMSE', 'R2']


,Model,MAE,RMSE,R2
0,Gradient Boosting 200,15.932161,21.423562,0.117053
1,Gradient Boosting 100,15.953633,21.424681,0.116961
2,Gradient Boosting Depth 3,15.961878,21.419572,0.117382
3,XGBoost 500,15.991414,21.470608,0.113171
4,Random Forest Depth 10,16.002567,21.475437,0.112772
5,XGBoost 300,16.064733,21.554957,0.106190
6,Ridge Alpha 1000,16.078629,21.447241,0.115101
7,Ridge Alpha 3000,16.085646,21.445985,0.115204
8,Ridge Alpha 100,16.105206,21.452342,0.114680
9,Ridge Alpha 10000,16.118473,21.457059,0.114290



FILE: batting_model_comparison.csv
STATUS: FOUND
Shape: (15, 4)
Columns: ['Model', 'MAE', 'RMSE', 'R2']


,Model,MAE,RMSE,R2
0,Random Forest Depth 10,16.002567,21.475437,0.112772
1,Ridge Alpha 1000,16.078629,21.447241,0.115101
2,Ridge Alpha 3000,16.085646,21.445985,0.115204
3,Ridge Alpha 100,16.105206,21.452342,0.114680
4,Ridge Alpha 10000,16.118473,21.457059,0.114290
5,Ridge Alpha 10,16.158372,21.463845,0.113730
6,Random Forest Depth 15,16.162721,21.562778,0.105541
7,Ridge Alpha 1,16.188406,21.477762,0.112580
8,Ridge Alpha 0.1,16.192793,21.480128,0.112385
9,Ridge Alpha 0.01,16.193257,21.480379,0.112364
